# 01 · GARCH foundations
## Predicting the Left Tail

How much can we learn about downside risk from a volatility model?

This is our working notebook for **Part 1, Days 1–2**. We'll move through the explanations and code together. Code cells are unexecuted; comments mark the pieces we'll write as we go.

Our scope: inspect the daily data, understand returns and squared returns, fit ARCH, GARCH, GARCH-t, and GJR-GARCH-t, then compare their responses to positive and negative shocks. Five-day forecasting and machine learning come later. We pause for review at the end of Part 1.

### 1. Get oriented

Select the project's `.venv/bin/python` as the notebook kernel. It uses Python 3.11.14. The notebook lives in `notebooks/`; the source code, data, and tests sit one directory above it.

We'll start with imports and locate the project directory. Nothing here downloads data or fits a model.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'pyproject.toml').exists()
)
print(ROOT)
print(sys.version)

### 2. Where should we get the data?

For this first experiment:

- **SPY:** Yahoo Finance through `yfinance`. We explicitly select **Adj Close**, which accounts for splits and dividends. That avoids treating a dividend adjustment as a market loss.
- **VIX:** Yahoo Finance's `^VIX` closing values.
- **VIX3M:** [Cboe's historical CSV](https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX3M_History.csv). Yahoo returned only one observation, so the supporting downloader used this source instead.

A local snapshot has already been downloaded into `data/raw/`. Its manifest records source details, dates, missingness, and checksums. We'll inspect that snapshot rather than fetch fresh data every time we run a cell.

VIX and VIX3M describe volatility implied by options over different horizons. We collect them now for later experiments; the Part 1 models use SPY returns only.

[The yfinance download options](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html).

In [ ]:
# Load the saved SPY data. First inspect the available columns and dates.
spy_data = pd.read_csv(
    ROOT / 'data/raw/SPY_source.csv',
    index_col='Date',
    parse_dates=True,
)
spy_data.head()

Before calculating anything, let's check the sample. What are the first and last dates? Are dates ordered and unique? Are prices missing or nonpositive?

Missing source rows and missing trading days are different checks. The current audit compares index coverage with the supplied SPY dates; it does not independently verify every exchange session. We should keep that limitation visible.

VIX3M begins later. We won't discard early SPY history just to force all three series to start together.

In [ ]:
# Together: inspect SPY coverage and missing adjusted prices.
# Then open data/raw/manifest.json to compare VIX and VIX3M coverage.
# Report the exact usable sample rather than guessing from the date range.

### 3. Calculate a daily return

We'll use **simple returns**:

$$r_t = \frac{P_t}{P_{t-1}} - 1.$$

If a price moves from 100 to 110, the return is 0.10, or 10%. What happens when it then falls from 110 to 99?

Let's calculate that small example before using SPY. The first observation has no previous price. Missing prices should stay missing, so we'll use `pct_change(fill_method=None)` rather than filling them.

In [ ]:
toy_prices = pd.Series([100.0, 110.0, 99.0])

# Together: calculate the returns and check each value by hand.
# Then apply the same calculation to spy_data['Adj Close'].

### 4. Why square returns?

Positive and negative returns can cancel in a sum. Squaring removes the sign and puts more weight on larger moves: a 4% move contributes four times as much as a 2% move.

Absolute values also prevent cancellation. Squaring is useful here because it connects the calculation to variance.

Try the returns below. How do the signed sum, absolute sum, and squared sum differ?

In [ ]:
toy_returns = pd.Series([0.04, -0.04, 0.02, -0.02])

# Together: compare sum(r), sum(abs(r)), and sum(r**2).
# Keep track of units: returns versus squared returns.

### 5. Are we assuming the expected return is zero?

Variance measures distance from the mean, while raw squared returns measure distance from zero:

$$\operatorname{Var}(r)=E[(r-\mu)^2],$$
$$E[r^2]=\operatorname{Var}(r)+\mu^2.$$

Here $\mu=E[r]$ is the expected return. Treating the mean squared return as variance uses a zero-mean assumption or approximation. We can inspect the size of that difference instead of assuming it away.

For a finite-sample demonstration, use `var(ddof=0)` so both quantities divide by the same observation count. A sample mean is an estimate, not a known future expected return.

For GARCH, we'll estimate a constant mean. The squared inputs are **shocks around that mean**, not raw returns.

In [ ]:
nonzero_mean_example = pd.Series([0.01, 0.02, -0.01, 0.03])

# Together: calculate the mean, mean squared return, and variance with ddof=0.
# Check that mean(r**2) equals var(r, ddof=0) + mean(r)**2.

### 6. Keep only the negative days

Daily downside variance is:

$$r_t^2 I(r_t<0).$$

The indicator $I$ is one if the return is negative and zero otherwise. Zero is our loss threshold; it does not require the expected return to be zero.

We're measuring squared negative movement, not the week's net loss or the variance of just the negative observations around their own mean. A later recovery won't cancel an earlier decline in this measure.

The eventual target will add these values over **t+1 through t+5**. Today we'll only build and test the daily measure.

In [ ]:
daily_example = pd.Series([0.04, -0.04, 0.0, -0.02, np.nan])

# Together: construct daily total squared returns and daily downside variance.
# What should happen to the positive day, the zero return, and the missing value?

### 7. Choose the training sample

We'll use returns through **2019-12-31** for the foundation fits and reserve later dates for future evaluation. This is a chronological split: earlier observations train the model, later observations remain outside the fit.

First report the full return count, training count, reserved count, and their date ranges. Then check how much the squared sample mean contributes to the training sample's mean squared return.

There are no forward labels yet. When we build them in Part 2, we'll also need to keep training labels from extending into the evaluation period.

In [ ]:
TRAIN_END = '2019-12-31'

# Together: split the SPY return series by date.
# Check ordering, boundary dates, sample sizes, and missing values.
# Use only the training returns in the model-fitting cells below.

### 8. Start with ARCH(1)

Write a return as an expected part plus a shock:

$$r_t=\mu+\epsilon_t.$$

ARCH(1) makes today's conditional variance depend on yesterday's squared shock:

$$\sigma_t^2=\omega+\alpha\epsilon_{t-1}^2.$$

“Conditional” means given the information already available. **Omega** is a baseline term; **alpha** controls the response to the latest squared shock.

We'll fit in percent-return units: 0.01 becomes 1.0. That means the fitted mean and shocks are in percent, while omega and variance are in percent squared. A fitted variance must be divided by 10,000 to get decimal-squared units.

Let's inspect the model specification before fitting it. What does choosing `mean='Constant'` ask the model to estimate?

In [ ]:
# Together: convert training returns from decimal to percent units.
# Specify arch_model(..., mean='Constant', vol='ARCH', p=1, dist='normal').
# Fit it, read the summary, and identify mu, omega, and alpha.
# Record optimizer warnings and convergence status.

### 9. Add memory with GARCH(1,1)

GARCH adds the previous variance estimate:

$$\sigma_t^2=\omega+\alpha\epsilon_{t-1}^2+\beta\sigma_{t-1}^2.$$

**Beta** controls how much carries forward. Under the standard model assumptions, **alpha + beta** describes persistence: closer to one means the effect of a shock fades more slowly.

When the sum is below one, the model's long-run variance is $\omega/(1-\alpha-\beta)$. Omega alone is not that long-run variance.

Can we read the fitted coefficients and explain how a large surprise affects the next estimate? [Model equations](https://arch.readthedocs.io/en/latest/univariate/generated/arch.univariate.GARCH.html).

In [ ]:
# Together: specify GARCH with p=1, o=0, q=1 and normal errors.
# Fit on the same training returns.
# Inspect alpha, beta, their sum, and the number of estimated parameters.

### 10. Give extreme surprises more room: GARCH-t

Student-t errors allow heavier tails than normal errors. The **degrees of freedom**, usually labeled `nu`, control the tail weight: smaller values mean heavier tails. The `arch` model uses a version standardized to unit variance, with nu above two.

This changes the assumed shock distribution. It does not make the variance response different for positive and negative shocks.

We'll change only `dist='StudentsT'` and compare the estimates. [Package examples](https://bashtage.github.io/arch/univariate/univariate_volatility_modeling.html).

In [ ]:
# Together: fit GARCH(1,1) with Student-t errors.
# Find nu. Compare alpha, beta, and the fitted mean with the normal-error model.
# How many parameters did we add?

### 11. Does a negative shock have a different effect?

Ordinary GARCH squares its shocks, so +4% and −4% have the same direct effect. These are deviations from the fitted mean; they equal raw returns only if that mean is zero.

GJR-GARCH adds a term for negative shocks:

$$\sigma_t^2=\omega+\alpha\epsilon_{t-1}^2
+\gamma\epsilon_{t-1}^2 I(\epsilon_{t-1}<0)
+\beta\sigma_{t-1}^2.$$

Positive squared shocks receive coefficient alpha. Negative squared shocks receive alpha plus **gamma**. With symmetric standardized errors, persistence becomes **alpha + beta + gamma/2**.

We'll estimate gamma and examine the result, including any coefficients near their constraints.

In [ ]:
# Together: fit GJR-GARCH-t with p=1, o=1, q=1, dist='StudentsT'.
# Compare alpha with alpha + gamma.
# Calculate persistence using the asymmetric model's formula.
# Inspect convergence and any boundary estimates.

### 12. Compare +1% with −1%, +2% with −2%, and +4% with −4%

Let's give the models the same previous variance and change only the shock. Use a starting variance calculated from the training sample, held fixed for every comparison.

Before drawing the figure: which responses should be identical? How would a positive gamma change the negative-shock response?

We'll plot the square root of the next-day conditional variance in daily percent units. This is a controlled one-step response comparison, not a five-day downside forecast. Compare GARCH-t with GJR-GARCH-t so both use the same error family.

In [ ]:
shock_sizes_percent = np.array([1.0, 2.0, 4.0])

# Together: implement one step of the variance equations.
# Evaluate positive and negative shocks at the same prior variance.
# Check the values, then draw and export the comparison figure.

### 13. How much data did these fits use?

Let's assemble a small table: model, training observations, reserved test observations, parameter count, and optimizer status. Include the estimated mean and Student-t degrees of freedom in the parameter count.

Several thousand daily observations may be informative for a model with a few parameters. The raw count still leaves questions about dependence, changing market conditions, and how many extreme episodes the sample contains.

We can compare in-sample fit, but we haven't tested future forecasts. A fitted volatility path uses parameters estimated from the entire training period; it must not be reused as if those estimates were available at every earlier date.

In [ ]:
# Together: build the sample-size and parameter-count table.
# Record fitting warnings, if any.
# Write a few sentences about what the estimates tell us and what remains open.

### 14. Check our work, then pause

As we implement the calculations, we'll check them against small examples and the tests in `tests/`. The supporting code already has tests we can read alongside our notebook versions.

By the end of Part 1, we want the data audit, four fitted models, the shock comparison, explanations of the parameters, passing tests, and an updated article and research log.

**Stop for review before Part 2.** The next research question will be how to connect these total-variance forecasts with our future downside target.

In [ ]:
# Together, once the notebook calculations are implemented:
# run the research tests and record their results.
# Update article/medium_draft.md and RESEARCH_LOG.md with our observations.